# Stage 3 of 3 — Optimize & Analyze

**This notebook does ONE thing**: apply the smell detector + exact-fidelity
verified repair (`rules.json` from Stage 2) to a broad sample of circuits
drawn from many chunk files, and report per-rule accept/quarantine telemetry.
This is the cheap, seconds-not-minutes stage — the one actually worth
iterating on directly, which is the whole point of splitting it out.

**Setup — two separate inputs:**
1. **Add Input** → search `mnisq-optbench-pairs` → Add (for sample circuits to
   test against — same as Stage 1).
2. **Add Input → Notebook Output → (your username) → `02-rule-building`** (the
   saved version of Stage 2, for `rules.json`).
3. **Internet: On**.

No mining, no rule-building happens here — if either input is missing, the
cells below fail fast with a clear message telling you which one.

In [ ]:
import glob
print(glob.glob('/kaggle/input/*'))
print(glob.glob('/kaggle/input/**/*', recursive=True)[:20])

In [ ]:
# Base package only -- same as Stage 2, no [mining] extra needed here.
!pip install -q "quantum-circuit-smell-intelligence @ git+https://github.com/veerakrish/quantum-circuit-smell-intelligence.git"

In [ ]:
try:
    import qcs_pipeline
    print("qcs_pipeline imported OK from:", qcs_pipeline.__file__)
except ModuleNotFoundError as e:
    raise RuntimeError(
        "qcs_pipeline is not importable. Check the pip install cell's output "
        "above, or restart the session if you just picked up a code update."
    ) from e

In [ ]:
import glob
from pathlib import Path

rule_matches = glob.glob("/kaggle/input/**/rules.json", recursive=True)
if not rule_matches:
    raise FileNotFoundError(
        "rules.json not found under /kaggle/input. Did you attach Stage 2's "
        "saved notebook output? (Add Input -> Notebook Output -> your "
        "username -> 02-rule-building)"
    )
RULES_PATH = Path(rule_matches[0])
print(f"Using {RULES_PATH}")

from qcs_pipeline.mining.from_kaggle_pairs import find_pair_chunks
try:
    chunk_files = find_pair_chunks(Path("/kaggle/input"))
except FileNotFoundError as e:
    raise FileNotFoundError(
        "mnisq-optbench-pairs dataset not found under /kaggle/input. Did you "
        "attach it? (Add Input -> search mnisq-optbench-pairs -> Add)"
    ) from e
print(f"Using {len(chunk_files)} sample-circuit chunk files, e.g. {chunk_files[0]}")

In [ ]:
import pandas as pd

# Sample across MANY chunk files instead of just chunk_files[0] -- a 5-circuit
# sample from a single chunk only ever exercised 1 of the 40 "applicable"
# rules from Stage 2 (the other 39 patterns simply never occurred in those 5
# circuits), so the previous run's 0-accepted/206-quarantined result was a
# coverage artifact, not a verdict on the whole rule database. Drawing from
# many chunks (which cover different circuits/labels) and enough rows per
# chunk is what actually exercises the rest of the database.
N_CHUNKS = 15
ROWS_PER_CHUNK = 25  # ~15 x 25 = 375 circuits total

frames = [
    pd.read_parquet(chunk_path, columns=["base_id", "input_qasm"]).head(ROWS_PER_CHUNK)
    for chunk_path in chunk_files[:N_CHUNKS]
]
df_sample = pd.concat(frames, ignore_index=True)
print(f"Sampling {len(df_sample)} circuits across {N_CHUNKS} chunk files ({ROWS_PER_CHUNK} rows/chunk)")
df_sample.head()

In [ ]:
# The only real work this notebook does: run the verified optimizer over the
# sampled circuits and aggregate PER-RULE accept/quarantine telemetry -- not
# just an overall pass/fail, but which specific rules (by pattern+rewrite
# identity, see rule_database.rule_id()) actually hold up under real
# exact-fidelity verification versus which ones only looked consistent during
# mining. A rule that's quarantined every time it's attempted is a template
# artifact of this dataset's shared circuit structure, not a true identity;
# a rule accepted every time it's attempted is a real, generalizable rewrite.
from collections import Counter

from qcs_pipeline.pipeline import QuantumCircuitSmellOptimizer

optimizer = QuantumCircuitSmellOptimizer(rule_db_path=RULES_PATH)

accept_counts: Counter[str] = Counter()
quarantine_counts: Counter[str] = Counter()
n_gates_before_total = 0
n_gates_after_total = 0
n_circuits_improved = 0
fidelities = []

for row in df_sample.itertuples(index=False):
    result = optimizer.optimize(row.input_qasm)
    fidelities.append(result.fidelity)
    n_gates_before_total += result.n_gates_before
    n_gates_after_total += result.n_gates_after
    if result.n_gates_after < result.n_gates_before:
        n_circuits_improved += 1
    for smell in result.applied_smells:
        accept_counts[smell.rule_id] += 1
    for rid, _reason in result.quarantined_rules:
        quarantine_counts[rid] += 1

n = len(df_sample)
min_fidelity = min(fidelities) if fidelities else float("nan")
total_accepted = sum(accept_counts.values())
total_quarantined = sum(quarantine_counts.values())

print(f"{n} circuits tested")
print(f"Gates: {n_gates_before_total} -> {n_gates_after_total} total "
      f"({n_gates_before_total - n_gates_after_total} removed)")
print(f"Circuits with >=1 gate removed: {n_circuits_improved}/{n}")
print(f"Min fidelity across all circuits: {min_fidelity:.10f}")
print(f"Total rule applications: {total_accepted} accepted, {total_quarantined} quarantined")

print("\nPer-rule breakdown, sorted by total attempts (accept + quarantine):")
all_rule_ids = set(accept_counts) | set(quarantine_counts)
rows = []
for rid in all_rule_ids:
    a, q = accept_counts[rid], quarantine_counts[rid]
    rows.append((a + q, a, q, a / (a + q) if (a + q) else 0.0, rid))
rows.sort(reverse=True)
for total, a, q, rate, rid in rows:
    print(f"  {total:4d} attempts | {a:4d} accepted | {q:4d} quarantined | pass_rate={rate:6.1%} | {rid}")
if not rows:
    print("  (no rule ever matched any sampled circuit -- see Interpreting results below)")

## Interpreting results

- **`Min fidelity` should read `1.0000000000` (or extremely close, floating-point
  noise).** That's the exact-fidelity verification working — any rule
  application that would have broken it gets bisected out and quarantined
  instead, never silently applied. This holds regardless of how good or bad
  the rule database is; it's the correctness guarantee, not a quality metric.
- **The per-rule breakdown is the actual quality signal.** `pass_rate=100.0%`
  on a rule seen many times means a real, generalizable identity — safe to
  trust and to keep even if `min_frequency` gets raised later. `pass_rate=0.0%`
  means every attempt failed the real check: that rule was only *syntactically*
  consistent across Stage 2's mining (same pattern, same resolved param
  relation every time it was observed), which is necessary but not sufficient
  for being a true, context-independent gate identity — it's more likely a
  template artifact specific to how this dataset's circuits are generated.
  Partial pass rates (neither 0% nor 100%) are the most interesting case: the
  rule holds in *some* contexts and not others, meaning the mined pattern is
  underspecified (missing some piece of context that determines whether the
  rewrite is actually valid).
- **A rule with 0 total attempts never matched any sampled circuit at all** —
  increase `N_CHUNKS`/`ROWS_PER_CHUNK` in the sampling cell above to get
  more coverage before drawing conclusions about it.